# 企业经营平台一年可行性演算

日期：2026-08-24  
决策：判断 F-57 在“现有 P340 硬件 + 已冻结安全边界”下运行一年的容量可行性，并区分容量、性能、恢复、开发周期和商业目标。  
读者：产品负责人、技术负责人、生产运维与安全审批人。

## tl;dr

- **运行容量：有条件可行。** 基准假设下，第 12 月 HDD 占用约 364 GiB、剩余约 567 GiB，全年保持绿色；附件重载压力档第 9 月进入黄色、第 12 月进入红色。
- **当前生产：不可放行。** 这是容量演算，不是 72 小时实机容量证书。单盘、UPS、服务器外只追加备份、两块离线盘、分域恢复材料、洁净恢复主机和恢复演练仍必须逐项取证。
- **一年开发：完整 Task 25 发布对“单人 + Codex”过于激进。** 25 个顺序任务在 52 周内平均每 2.08 周关闭一个，还要覆盖四端、Windows 实机、恢复、72 小时稳定负载和最终聚合。合理目标是可控试点；完整认证需要小型跨职能团队和独立复核。
- **年入 1 亿：是成熟期规模算式，不是第一年预测。** 300 家 × 32 万年费 + 50 次 × 8 万启用费 = 1 亿元；从零代码出发，第一年应先证明 1–3 个设计伙伴和 3–10 个付费试点的可复制交付。

## Context & Methods

### Key Assumptions

1. 数据盘标称 1 TB，按二进制口径换算为约 931.3 GiB；所有客户及衍生持久数据都在 HDD，SSD 只放系统和可重建程序。
2. 并发认证包络为 15 个 Workbench、3 个客户门户、2 个供应商门户，另有 1 个独立资源的 Control Center 会话。
3. 不运行本地模型，不在当前 32 GB P340 上启用可选 Hyper-V 容器；大型报表后台并发为 1。
4. 三档均为规划假设，不是客户实测：轻载、基准、附件重载压力。附件、审计、日志和 WAL 的真实增长需要上线前样本重取。
5. 磁盘阈值完全沿用 F-57：`emergency_reserve=max(20 GiB, capacity×5%)`；`yellow_free=max(2×reserve, P95 日增长×30)`；`red_free=reserve`。
6. 日增长用固定随机种子的对数正态形状分配到 260 个工作日，并在每月归一到年度假设；它只用于得到可复算的 P95 压力形状，不代表真实生产分布。
7. 离线介质容量下限采用“年末可恢复集 + 20% 加密/校验/恢复工作空间 + 30 日 P95 增长”。20% 是本演算假设，正式证书必须用实际工具测量。
8. 性能、RPO、RTO 和抗勒索结果不能由纸面计算证明，统一保持 `UNVERIFIED`，直到 Windows/P340、真实 PostgreSQL、真实介质和洁净主机演练通过。

In [1]:
from __future__ import annotations

import math
from dataclasses import dataclass

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

CAPACITY_GIB = 1_000_000_000_000 / 1024**3
EMERGENCY_RESERVE_GIB = max(20.0, CAPACITY_GIB * 0.05)
MONTHS = [f"M{i}" for i in range(1, 13)]
BUSINESS_DAYS = np.array([22, 20, 22, 22, 22, 21, 23, 22, 22, 22, 21, 21])
MONTH_WEIGHTS = np.array([0.07, 0.07, 0.08, 0.08, 0.08, 0.08, 0.08, 0.08, 0.12, 0.08, 0.09, 0.09])

assert BUSINESS_DAYS.sum() == 260
assert math.isclose(MONTH_WEIGHTS.sum(), 1.0)

SCENARIOS = {
    "轻载": {
        "start_gib": 35.0,
        "working_headroom_gib": 45.0,
        "structured_gib_year": 18.0,
        "audit_search_gib_year": 24.0,
        "attachments_gib_year": 60.0,
        "retained_ops_gib_year": 12.0,
        "attachment_count": 30_000,
        "seed": 5700,
    },
    "基准": {
        "start_gib": 40.0,
        "working_headroom_gib": 60.0,
        "structured_gib_year": 36.0,
        "audit_search_gib_year": 48.0,
        "attachments_gib_year": 150.0,
        "retained_ops_gib_year": 30.0,
        "attachment_count": 60_000,
        "seed": 5701,
    },
    "附件重载压力": {
        "start_gib": 40.0,
        "working_headroom_gib": 80.0,
        "structured_gib_year": 60.0,
        "audit_search_gib_year": 90.0,
        "attachments_gib_year": 600.0,
        "retained_ops_gib_year": 60.0,
        "attachment_count": 120_000,
        "seed": 5702,
    },
}

## Data

基准档按 20 名活跃用户的一家合同驱动型企业建模：约 2 个法人、50 个命名用户、2,400 份报价、720 份合同、1,800 个销售订单、1,500 个采购订单、18 万条库存/履约移动、50 万条经营分录、2,400 张售后工单、约 500 万条审计/自动化/消息状态行，以及 6 万个平均 2.56 MiB 的附件。业务数量用于说明量级；容量公式直接使用下表各类年度净增量。

In [2]:
business_volume = pd.DataFrame(
    [
        ("法人", 2, "个"),
        ("命名用户", 50, "人"),
        ("同时活跃用户", 20, "人"),
        ("报价", 2_400, "份/年"),
        ("合同", 720, "份/年"),
        ("销售订单", 1_800, "单/年"),
        ("采购订单", 1_500, "单/年"),
        ("库存/履约移动", 180_000, "条/年"),
        ("经营分录", 500_000, "条/年"),
        ("售后工单", 2_400, "张/年"),
        ("审计/自动化/消息状态", 5_000_000, "行/年"),
        ("附件", 60_000, "个/年"),
    ],
    columns=["业务对象", "基准数量", "单位"],
)
business_volume

,业务对象,基准数量,单位
0,法人,2,个
1,命名用户,50,人
2,同时活跃用户,20,人
3,报价,2400,份/年
4,合同,720,份/年
5,销售订单,1800,单/年
6,采购订单,1500,单/年
7,库存/履约移动,180000,条/年
8,经营分录,500000,条/年
9,售后工单,2400,张/年


In [3]:
scenario_rows = []
for name, cfg in SCENARIOS.items():
    annual_growth = sum(
        cfg[key]
        for key in (
            "structured_gib_year",
            "audit_search_gib_year",
            "attachments_gib_year",
            "retained_ops_gib_year",
        )
    )
    scenario_rows.append(
        {
            "场景": name,
            "初始业务数据_GiB": cfg["start_gib"],
            "运行工作空间_GiB": cfg["working_headroom_gib"],
            "结构化数据年增_GiB": cfg["structured_gib_year"],
            "审计搜索年增_GiB": cfg["audit_search_gib_year"],
            "附件年增_GiB": cfg["attachments_gib_year"],
            "日志导出年增_GiB": cfg["retained_ops_gib_year"],
            "附件数": cfg["attachment_count"],
            "平均附件_MiB": cfg["attachments_gib_year"] * 1024 / cfg["attachment_count"],
            "年度净增_GiB": annual_growth,
        }
    )

assumption_table = pd.DataFrame(scenario_rows)
assumption_table

,场景,初始业务数据_GiB,运行工作空间_GiB,结构化数据年增_GiB,审计搜索年增_GiB,附件年增_GiB,日志导出年增_GiB,附件数,平均附件_MiB,年度净增_GiB
0,轻载,35.00,45.00,18.00,24.00,60.00,12.00,30000,2.05,114.00
1,基准,40.00,60.00,36.00,48.00,150.00,30.00,60000,2.56,264.00
2,附件重载压力,40.00,80.00,60.00,90.00,600.00,60.00,120000,5.12,810.00


In [4]:
daily_rows = []
monthly_rows = []
summary_rows = []

for scenario_name, cfg in SCENARIOS.items():
    annual_growth = (
        cfg["structured_gib_year"]
        + cfg["audit_search_gib_year"]
        + cfg["attachments_gib_year"]
        + cfg["retained_ops_gib_year"]
    )
    rng = np.random.default_rng(cfg["seed"])
    daily_growth = []
    day_of_year = 0
    for month_index, (month, day_count, weight) in enumerate(
        zip(MONTHS, BUSINESS_DAYS, MONTH_WEIGHTS), start=1
    ):
        shape = rng.lognormal(mean=0.0, sigma=0.55, size=int(day_count))
        month_values = shape / shape.sum() * annual_growth * weight
        for day_in_month, value in enumerate(month_values, start=1):
            day_of_year += 1
            daily_rows.append(
                {
                    "场景": scenario_name,
                    "月份": month,
                    "月序": month_index,
                    "月内工作日": day_in_month,
                    "全年工作日": day_of_year,
                    "净增长_GiB": float(value),
                }
            )
            daily_growth.append(float(value))

    p95_daily_growth = float(np.quantile(daily_growth, 0.95))
    yellow_free = max(EMERGENCY_RESERVE_GIB * 2, p95_daily_growth * 30)
    red_free = EMERGENCY_RESERVE_GIB
    used = cfg["start_gib"] + cfg["working_headroom_gib"]
    first_yellow = None
    first_red = None

    for month_index, (month, weight) in enumerate(zip(MONTHS, MONTH_WEIGHTS), start=1):
        used += annual_growth * weight
        free = CAPACITY_GIB - used
        status = "RED" if free < red_free else "YELLOW" if free < yellow_free else "GREEN"
        if status == "YELLOW" and first_yellow is None:
            first_yellow = month_index
        if status == "RED" and first_red is None:
            first_red = month_index
        monthly_rows.append(
            {
                "场景": scenario_name,
                "月份": month,
                "月序": month_index,
                "占用_GiB": used,
                "剩余_GiB": free,
                "占用率": used / CAPACITY_GIB,
                "黄色剩余线_GiB": yellow_free,
                "红色剩余线_GiB": red_free,
                "状态": status,
            }
        )

    recoverable_set = cfg["start_gib"] + annual_growth
    offline_media_floor = recoverable_set * 1.20 + p95_daily_growth * 30
    end_used = cfg["start_gib"] + cfg["working_headroom_gib"] + annual_growth
    end_free = CAPACITY_GIB - end_used
    end_status = "RED" if end_free < red_free else "YELLOW" if end_free < yellow_free else "GREEN"
    summary_rows.append(
        {
            "场景": scenario_name,
            "年末占用_GiB": end_used,
            "年末剩余_GiB": end_free,
            "年末占用率": end_used / CAPACITY_GIB,
            "P95日增长_GiB": p95_daily_growth,
            "黄色剩余线_GiB": yellow_free,
            "红色剩余线_GiB": red_free,
            "首次黄色月": first_yellow,
            "首次红色月": first_red,
            "年末状态": end_status,
            "单块离线介质最低可用_GiB_演算": offline_media_floor,
        }
    )

daily = pd.DataFrame(daily_rows)
monthly = pd.DataFrame(monthly_rows)
summary = pd.DataFrame(summary_rows)
summary

,场景,年末占用_GiB,年末剩余_GiB,年末占用率,P95日增长_GiB,黄色剩余线_GiB,红色剩余线_GiB,首次黄色月,首次红色月,年末状态,单块离线介质最低可用_GiB_演算
0,轻载,194.00,737.32,0.21,0.96,93.13,46.57,NaN,NaN,GREEN,207.72
1,基准,364.00,567.32,0.39,2.19,93.13,46.57,NaN,NaN,GREEN,430.43
2,附件重载压力,930.00,1.32,1.00,7.20,216.02,46.57,9.00,12.00,RED,"1,236.02"


## Results

### 容量结果

轻载和基准档全年保持绿色。压力档的真正问题不是第 12 月才“突然满盘”，而是第 9 月就必须停止大型导入、导出、报表和非必要索引重建；如果仍继续增长，第 12 月会进入只能完成在途事务、审计、安全处置和受控停机的红色区。

In [5]:
monthly_pivot = monthly.pivot(index="月份", columns="场景", values="占用_GiB").reindex(MONTHS)
monthly_pivot

场景,基准,轻载,附件重载压力
月份,,,
M1,118.48,87.98,176.70
M2,136.96,95.96,233.40
M3,158.08,105.08,298.20
M4,179.20,114.20,363.00
M5,200.32,123.32,427.80
M6,221.44,132.44,492.60
M7,242.56,141.56,557.40
M8,263.68,150.68,622.20
M9,295.36,164.36,719.40


In [6]:
baseline_other_end_gib = (
    SCENARIOS["基准"]["start_gib"]
    + SCENARIOS["基准"]["working_headroom_gib"]
    + SCENARIOS["基准"]["structured_gib_year"]
    + SCENARIOS["基准"]["audit_search_gib_year"]
    + SCENARIOS["基准"]["retained_ops_gib_year"]
)
baseline_p95_ratio = (
    summary.loc[summary["场景"] == "基准", "P95日增长_GiB"].iloc[0]
    / (summary.loc[summary["场景"] == "基准", "年末占用_GiB"].iloc[0] - 100.0)
    * 260
)

sensitivity_rows = []
for attachments_gib in range(0, 701, 25):
    annual_growth = 114.0 + attachments_gib
    p95_est = annual_growth / 260 * baseline_p95_ratio
    yellow_free = max(EMERGENCY_RESERVE_GIB * 2, p95_est * 30)
    end_used = baseline_other_end_gib + attachments_gib
    end_free = CAPACITY_GIB - end_used
    status = "RED" if end_free < EMERGENCY_RESERVE_GIB else "YELLOW" if end_free < yellow_free else "GREEN"
    sensitivity_rows.append(
        {
            "附件年增_GiB": attachments_gib,
            "年末占用_GiB": end_used,
            "年末剩余_GiB": end_free,
            "估算黄色线_GiB": yellow_free,
            "状态": status,
        }
    )
sensitivity = pd.DataFrame(sensitivity_rows)
last_green = sensitivity[sensitivity["状态"] == "GREEN"].iloc[-1]
first_yellow_sensitivity = sensitivity[sensitivity["状态"] == "YELLOW"].iloc[0]
first_red_sensitivity = sensitivity[sensitivity["状态"] == "RED"].iloc[0]

pd.DataFrame([last_green, first_yellow_sensitivity, first_red_sensitivity])

,附件年增_GiB,年末占用_GiB,年末剩余_GiB,估算黄色线_GiB,状态
22,550,764.00,167.32,165.06,GREEN
23,575,789.00,142.32,171.27,YELLOW
27,675,889.00,42.32,196.13,RED


### 附件是容量总开关

在其他基准项不变时，25 GiB 步长敏感性结果显示：附件年增 550 GiB 仍勉强绿色，575 GiB 进入黄色，675 GiB 进入红色。按每年 6 万个附件换算，绿色边界约等于平均每个附件 9–10 MiB。该数字只是采购前规划线；正式门槛必须用真实文件大小分布、压缩包展开、版本留存、隔离区、WAL 和索引增长重算。

### 恢复容量比本机“剩余空间”更严格

基准档单块离线介质的演算下限约 431 GiB；压力档约 1,236 GiB，已经超过标称 1 TB HDD 的约 931 GiB 可用空间。并且连续只追加目标还要容纳多个签名代际与至少 90 天抗勒索保留，不能直接把这里的单介质下限当作连续目标采购容量。

In [7]:
recovery_capacity = summary[["场景", "单块离线介质最低可用_GiB_演算"]].copy()
recovery_capacity["标称1TB单盘是否装得下"] = recovery_capacity["单块离线介质最低可用_GiB_演算"].le(CAPACITY_GIB)
recovery_capacity

,场景,单块离线介质最低可用_GiB_演算,标称1TB单盘是否装得下
0,轻载,207.72,True
1,基准,430.43,True
2,附件重载压力,"1,236.02",False


### 性能不能靠容量公式宣布通过

CPU 的 6 核 12 线程对 20 人普通业务是合理候选，32 GB 内存则是紧预算。单块旋转 HDD 同时承担 PostgreSQL data/WAL/temp、附件、审计、搜索索引、隔离区和备份暂存，随机 I/O 是首要瓶颈。正式判断必须在真实 P340 上同时运行 20 个不同 principal、1 个 Control Center、增量备份、闭环自动化、附件 burst 和 1 个大型报表，连续至少 72 小时，并证明普通读 P95 ≤2 秒、普通权威写 P95 ≤3 秒。

规划内存预算可用 28 GiB 为软上限，留 4 GiB 安全余量；超过即优先升级 64 GB，而不是削弱审计、备份或扫描。这个 28 GiB 是规划触发器，不是 F-57 已认证阈值。

In [8]:
memory_budget = pd.DataFrame(
    [
        ("Windows Server + 安全软件", 6.0),
        ("PostgreSQL 进程与缓存", 8.0),
        ("Rust 权威服务", 6.0),
        ("扫描器、控制中心、监控", 3.0),
        ("单报表/备份/插件瞬时", 5.0),
        ("必须保留的安全余量", 4.0),
    ],
    columns=["资源类别", "规划_GiB"],
)
memory_budget.loc[len(memory_budget)] = ["合计", memory_budget["规划_GiB"].sum()]
memory_budget

,资源类别,规划_GiB
0,Windows Server + 安全软件,6.00
1,PostgreSQL 进程与缓存,8.00
2,Rust 权威服务,6.00
3,扫描器、控制中心、监控,3.00
4,单报表/备份/插件瞬时,5.00
5,必须保留的安全余量,4.00
6,合计,32.00


### 十二个月故障与业务闭环演练

下列 12 个事件在设计中都有唯一安全结果，静态覆盖为 12/12；由于产品尚未实现，运行证据仍是 0/12。任何事件只有在真实代码、数据库、客户端和介质上重放通过后，才能改成 PASS。

In [9]:
year_events = pd.DataFrame(
    [
        (1, "上线与导入", "权限/法人/密钥域重验；客户字节只落 HDD；导入可恢复"),
        (2, "报价→合同→订单", "转换幂等；版本、来源、审批和审计可追溯"),
        (3, "采购与分批到货", "数量不超上游；部分收货、交付和退货可继续闭环"),
        (4, "期间锁定后迟到发票", "不重开已锁期间；顺延开放期间并保留更正链"),
        (5, "模块热升级时有长流程", "新请求见新代；在途流程钉住旧兼容代；失败可回滚"),
        (6, "第一次洁净恢复演练", "从服务器外、离线介质和分域材料恢复并对账"),
        (7, "员工离职/设备撤销/临时委派", "权限实时收回；委派有范围和到期；离线意图重验"),
        (8, "MCP/外部系统超时但可能已成功", "结果进入 Unknown；先对账，禁止盲目重做不可逆动作"),
        (9, "订单与附件双倍峰值", "交易优先；报表/导入排队；到黄色线自动降级"),
        (10, "扫描定义过期或恶意附件", "附件继续隔离；UNKNOWN/SKIPPED/超时一律不发布"),
        (11, "勒索演练与最新备份投毒", "fence 权威端；跳过污染代；洁净主机恢复已签 known-clean 代"),
        (12, "年结、完整导出与保留", "不可变更正；完整可验证导出；legal hold/retention 不被许可证破坏"),
    ],
    columns=["月份", "演练事件", "必须得到的结果"],
)
year_events["当前运行证据"] = "NOT_IMPLEMENTED"
year_events

,月份,演练事件,必须得到的结果,当前运行证据
0,1,上线与导入,权限/法人/密钥域重验；客户字节只落 HDD；导入可恢复,NOT_IMPLEMENTED
1,2,报价→合同→订单,转换幂等；版本、来源、审批和审计可追溯,NOT_IMPLEMENTED
2,3,采购与分批到货,数量不超上游；部分收货、交付和退货可继续闭环,NOT_IMPLEMENTED
3,4,期间锁定后迟到发票,不重开已锁期间；顺延开放期间并保留更正链,NOT_IMPLEMENTED
4,5,模块热升级时有长流程,新请求见新代；在途流程钉住旧兼容代；失败可回滚,NOT_IMPLEMENTED
5,6,第一次洁净恢复演练,从服务器外、离线介质和分域材料恢复并对账,NOT_IMPLEMENTED
6,7,员工离职/设备撤销/临时委派,权限实时收回；委派有范围和到期；离线意图重验,NOT_IMPLEMENTED
7,8,MCP/外部系统超时但可能已成功,结果进入 Unknown；先对账，禁止盲目重做不可逆动作,NOT_IMPLEMENTED
8,9,订单与附件双倍峰值,交易优先；报表/导入排队；到黄色线自动降级,NOT_IMPLEMENTED
9,10,扫描定义过期或恶意附件,附件继续隔离；UNKNOWN/SKIPPED/超时一律不发布,NOT_IMPLEMENTED


### 从零开发到一年后的现实边界

现行计划有 25 个严格顺序任务。52 周平均只有 2.08 周/任务，而且这还没有扣除架构返工、四端真机、签名与发布、Windows 驱动、72 小时 soak、恢复演练、渗透测试和试点修复。故：

- **单人 + Codex：**适合完成基础、关键纵向闭环和受控演示/试点，不应承诺一年内完成最高安全档 Task 25 商用认证。
- **低成本可行组织：**1 名强产品/架构负责人 + 3–5 名能跨 Rust/PostgreSQL/跨端/Windows 的工程成员，按需购买财务、安全、恢复和真机测试复核；仍要严格冻结首版范围。
- **一年最合理的终点：**先完成同一客户的一条 `客户→合同→订单→采购→交付→开票→收款→售后` 闭环，在 Q4 争取受控设计伙伴试点。完整发布若任一安全门未过，宁可顺延，不得把静态文档当认证。

In [10]:
development_math = pd.DataFrame(
    [
        ("现行顺序任务", 25, "个"),
        ("一年总周数", 52, "周"),
        ("平均每任务可用时间", 52 / 25, "周/任务"),
        ("最终实机稳定负载", 72, "小时"),
        ("生产客户端平台", 4, "个平台"),
    ],
    columns=["项目", "数值", "单位"],
)
development_math

,项目,数值,单位
0,现行顺序任务,25.00,个
1,一年总周数,52.00,周
2,平均每任务可用时间,2.08,周/任务
3,最终实机稳定负载,72.00,小时
4,生产客户端平台,4.00,个平台


### 一亿元目标是规模状态，不是第一年收入预测

按先前讨论过的定价算式，300 家客户 × 32 万元年费 = 9,600 万元；再加 50 次 × 8 万元启用费 = 400 万元，合计 1 亿元。算术成立，但没有任何数据证明第一年能获得 300 家企业客户。第一年更合理的商业验证是 1–3 家设计伙伴、3–10 个付费试点，证明每次部署、升级、恢复和行业定制都能被签名包与自动证据复用。

In [11]:
commercial_scale = pd.DataFrame(
    [
        (10, 320_000, 3_200_000),
        (50, 320_000, 16_000_000),
        (100, 320_000, 32_000_000),
        (300, 320_000, 96_000_000),
    ],
    columns=["客户数", "假设年费_元每客户", "年度许可收入_元"],
)
activation_revenue = 50 * 80_000
target_total = commercial_scale.iloc[-1]["年度许可收入_元"] + activation_revenue
commercial_scale, activation_revenue, target_total

(   客户数  假设年费_元每客户  年度许可收入_元
 0   10     320000   3200000
 1   50     320000  16000000
 2  100     320000  32000000
 3  300     320000  96000000,
 4000000,
 np.int64(100000000))

## Takeaways

1. **基准运行容量可行，但生产仍是 NO-GO。** 364 GiB 年末占用说明 1 TB HDD 在普通附件分布下有空间；它不能代替单盘冗余、备份与恢复门。
2. **先盯附件增量和 HDD 延迟。** 年附件增量接近 550 GiB 时应提前扩容；任一 P95 读写、WAL、队列或 SMART 指标异常，应在业务受影响前触发降级。
3. **最低成本升级顺序：**先补齐 UPS、服务器外只追加目标、两块离线盘、分域恢复材料和洁净恢复能力；随后优先 64 GB RAM 与两块匹配企业 CMR HDD 镜像。RAID1 只降低停机风险，不替代备份。
4. **一年开发以试点为目标。** 完整 Task 25 是发布门，不是必须硬塞进 12 个月的营销日期；先证明一个闭环和自动恢复，才能形成下一代护城河。
5. **1 亿元靠复制，不靠堆定制人力。** 真正支点是“行业包 + 自动部署 + 自动取证 + 可恢复长链条”，让第 100 个客户的交付成本显著低于第 1 个客户。

In [12]:
# 独立一致性断言：这些是报告中最高影响结论的机械复核。
baseline = summary.set_index("场景").loc["基准"]
pressure = summary.set_index("场景").loc["附件重载压力"]

assert round(CAPACITY_GIB, 1) == 931.3
assert math.isclose(baseline["年末占用_GiB"], 364.0)
assert round(baseline["年末剩余_GiB"], 1) == 567.3
assert baseline["年末状态"] == "GREEN"
assert int(pressure["首次黄色月"]) == 9
assert int(pressure["首次红色月"]) == 12
assert pressure["年末状态"] == "RED"
assert recovery_capacity.loc[
    recovery_capacity["场景"] == "附件重载压力", "标称1TB单盘是否装得下"
].item() is False or recovery_capacity.loc[
    recovery_capacity["场景"] == "附件重载压力", "标称1TB单盘是否装得下"
].item() == False
assert target_total == 100_000_000

print("VALIDATION_PASS")
print(f"容量={CAPACITY_GIB:.1f} GiB；基准年末={baseline['年末占用_GiB']:.1f} GiB；压力黄线=M{int(pressure['首次黄色月'])}；压力红线=M{int(pressure['首次红色月'])}")

VALIDATION_PASS
容量=931.3 GiB；基准年末=364.0 GiB；压力黄线=M9；压力红线=M12
